# Laboratorio 4 · ¿Quién decide la ruta? Modelo de decisión, modelo de lenguaje y umbral

**Objetivo:** decidir a qué equipo va el mensaje de un pasajero con un modelo de decisión (d1-2B) y con un modelo de lenguaje, medir con qué frecuencia acierta cada uno, y elegir un umbral de confianza que derive a una persona los casos inciertos, comprobado en casos que no se usaron para elegirlo.

Trabajen en el equipo del proyecto y alternen quién ejecuta y quién revisa. El laboratorio se ejecuta en **Google Colab** con una cuenta personal de Google; no necesitas cuenta de GitHub.

El producto compartido se guarda en `labs/lab04-decision/`; cada integrante conserva su reflexión y su historial. Descarguen los archivos antes de cerrar Colab: los archivos de la sesión son temporales. La pregunta individual se responde en Canvas al cierre de la clase del 06/11; producto, reflexión e historial tienen plazo hasta el **martes 10/11/2026 a las 23:59, hora de Chile**.

El sistema propone una ruta para atender el mensaje; no decide devoluciones, compensaciones ni derechos.

## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación (tarda alrededor de un minuto). Después agrega tu clave de Gemini como `GEMINI_API_KEY` o `GOOGLE_API_KEY` en **Secretos** (icono de llave de Colab) y permite que este notebook la lea. Nunca escribas su valor en una celda, captura o conversación.

En la celda de configuración, `MODE` puede ser `real` (consulta los modelos) o `simulated` (respuestas inventadas para practicar sin clave). Se abre en modo simulado; para la entrega cambia a `real` y vuelve a ejecutar.

**Dos modelos, dos roles.**

- **Modelo de decisión** (`DECISION_MODEL`): d1-2B, un modelo de 2 mil millones de parámetros que no escribe texto: recibe una pregunta con opciones fijas y devuelve una **probabilidad para cada opción**. En clase usa el servidor del curso (`d1:curso`), con `CURSO_API_KEY` y `CURSO_BASE_URL` en Secretos. Fuera de clase, `d1:colab` lo levanta en tu propia sesión con `llama-server` (sección 4).
- **Modelo de lenguaje** (`MODEL`): en clase, Gemma 4 12B del servidor del curso (`curso:course-chat`); fuera de clase, `google:gemini-3.5-flash-lite`, gratis con tu clave de Gemini. Indica en tu entrega cuál usaste para cada rol.

La tabla de la celda de configuración muestra cada rol, el modelo elegido, si está listo (`ready`) y qué secreto falta (`missing`).

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # fuera de Colab puede no existir el paquete google
    IN_COLAB = False

if IN_COLAB:
    PACKAGE_REF = "lab04"
    # Para el ensayo docente: un ZIP del paquete subido a la sesión reemplaza la etiqueta.
    LOCAL_PACKAGE = Path("/content/curso-ia-lab04.zip")
    package_source = (f"curso-ia @ file://{LOCAL_PACKAGE}" if LOCAL_PACKAGE.exists() else
                      f"curso-ia @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package_source])
    print("Paquete instalado. Continúa con los secretos y la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")

In [ ]:
import os
from typing import Literal

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from curso_ia import create_model, load_config, save_experiment, status
from curso_ia.decision import load_routes, reliability, retry_routes, run_routes, start_d1

# Cambia "simulated" por "real" para consultar los modelos con tus claves de Secretos.
MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE

# Modelo de decisión:
#   "d1:curso"  d1-2B en el servidor del curso, solo durante la clase; requiere CURSO_API_KEY y CURSO_BASE_URL.
#   "d1:colab"  d1-2B en tu sesión de Colab; la sección 4 lo descarga y lo levanta.
DECISION_MODEL = "d1:curso"  # @param ["d1:curso", "d1:colab"]
os.environ["DECISION_MODEL"] = DECISION_MODEL

# Modelo de lenguaje, con la forma proveedor:modelo:
#   "curso:course-chat"             servidor del curso con Gemma 4 12B, solo durante la clase.
#   "google:gemini-3.5-flash-lite"  Gemini, gratis con tu clave; para trabajar fuera de clase.
MODEL = "curso:course-chat"  # @param ["curso:course-chat", "google:gemini-3.5-flash-lite"]
os.environ["GENERATION_MODEL"] = MODEL

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT = ROOT / "labs/lab04-decision"
config = load_config(ROOT)
decision_ref, model_ref = config.models["decision"], config.models["generation"]
print("Modo:", config.mode, "· Modelo de decisión:", decision_ref, "· Modelo de lenguaje:", model_ref)
display(pd.DataFrame([row for row in status(config) if row["role"] in ("generation", "decision")]))

## 2. Conocer las rutas, las reglas y los casos

Pacífico Sur, la aerolínea de los laboratorios 2 y 3, recibe mensajes de pasajeros y debe enviarlos al equipo que los atiende. Hay **cinco rutas** y **cinco reglas** escritas por el docente, que se aplican en orden: gana la primera que se cumple.

El conjunto tiene **80 mensajes sintéticos** con la ruta correcta según esas reglas. Cada caso trae:

- `difficulty`: `clear` si una sola regla aplica sin dudas, `hard` si el mensaje toca varios temas, niega algo o pide información sin una solicitud.
- `split`: `tune` (41 casos) para elegir el umbral y `test` (39) para comprobarlo. Elegir y comprobar con los mismos casos premia un umbral que solo calza con ellos.
- `note`: en los casos difíciles, por qué la etiqueta es esa.

Una etiqueta puede ser discutible: si no están de acuerdo con una, anótenlo en `DECISIONS`; no la cambien en los datos.

In [ ]:
routes = load_routes()
cases = routes["cases"]
gold = pd.DataFrame(cases).set_index("id")

for name, description in routes["labels"].items():
    print(f"- {name}: {description}")
print()
print("\n".join(routes["rules"]))
display(pd.crosstab(gold["route"], [gold["split"], gold["difficulty"]], margins=True))

In [ ]:
# Tres casos difíciles, con la regla que decide y por qué; cambia la semilla para ver otros.
for case_id, case in gold[gold["difficulty"] == "hard"].sample(3, random_state=1).iterrows():
    print(f"{case_id} · ruta {case['route']} · regla {case['rule']}")
    print("  Mensaje:", case["text"])
    print("  Por qué:", case["note"])
    print()

## 3. Escribir la decisión como una pregunta tipada

Con PydanticAI, la misma clase sirve para los dos modelos. `Route` es un `Literal`: solo admite las cinco rutas. La descripción del campo es la pregunta y `INSTRUCTION` reúne las rutas y las reglas.

- Un **modelo de decisión** no redacta: el servidor lee la pregunta y las opciones, y devuelve la probabilidad de cada opción. La **confianza** de la ruta elegida sale de esas probabilidades: es un margen respecto de las demás opciones, no la probabilidad de acertar.
- Un **modelo de lenguaje** escribe la respuesta como JSON. Para poder barrer un umbral también con él, le pedimos un campo más, `confidence`, que **declara él mismo**. En el laboratorio 2 vimos que la confianza declarada no siempre significa algo; aquí lo medimos.

Cada campo del esquema es una pregunta para el modelo de decisión, y su tipo decide qué devuelve: un `bool`, la probabilidad de sí; un `Literal`, la de cada opción (lo que usamos aquí); una `list` de un `Literal`, un sí o no por opción; y una rúbrica de niveles 0, 1, 2… descritos, la de cada nivel. Un campo `str` no lo puede llenar: el texto libre lo escribe un modelo de lenguaje.

In [ ]:
Route = Literal["rebooking", "refund", "baggage", "special_assistance", "other"]


class RouteDecision(BaseModel):
    """Ruta del mensaje de un pasajero de Pacífico Sur."""

    route: Route = Field(description="¿A qué equipo debe ir este mensaje según las reglas?")


class RouteAnswer(RouteDecision):
    """La misma ruta, con la confianza que declara el modelo de lenguaje."""

    confidence: float = Field(ge=0, le=1, description="Qué tan seguro estás de la ruta, de 0 a 1")


INSTRUCTION = (
    "Clasificas mensajes de pasajeros de la aerolínea Pacífico Sur en una ruta.\n\nRutas:\n"
    + "\n".join(f"- {name}: {description}" for name, description in routes["labels"].items())
    + "\n\nReglas, en orden; gana la primera que se cumple:\n" + "\n".join(routes["rules"])
)
print(INSTRUCTION)

## 4. Levantar d1-2B en Colab (solo fuera de clase)

En clase, con `d1:curso`, salta esta sección.

Con `d1:colab`, la celda descarga `llama-server` de [llama.cpp](https://github.com/ggml-org/llama.cpp) y el modelo [d1-2B](https://huggingface.co/LiquidAI/d1-2B-GGUF) (1,9 GB en `Q8_0`) y lo deja corriendo en tu sesión. Tarda unos minutos la primera vez. Con GPU (**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**) debería tardar menos de un segundo por caso; sin GPU, unos 5 segundos, así que los 80 casos toman cerca de 7 minutos. `Q4_K_M` (1,1 GB) es más liviano: si lo usan, decláranlo.

d1 usa la [licencia LFM Open License v1.0](https://huggingface.co/LiquidAI/d1-2B/blob/main/LICENSE), que restringe el uso comercial según los ingresos de la entidad: revisarla es parte de elegir un modelo para un proyecto.

In [ ]:
D1_QUANT = "Q8_0"  # @param ["Q8_0", "Q4_K_M"]
if decision_ref == "d1:colab" and config.mode == "real":
    start_d1(D1_QUANT)
else:
    print("No hace falta: se usa", decision_ref, "en modo", config.mode)

## 5. Probar los dos agentes con un caso

Cada `Agent` combina un modelo, una salida tipada y la instrucción. Con d1 miramos las probabilidades de las cinco rutas: si dos están cerca, el modelo duda.

In [ ]:
decision_agent = Agent(create_model(decision_ref, config), output_type=RouteDecision, instructions=INSTRUCTION)
language_agent = Agent(create_model(model_ref, config), output_type=RouteAnswer, instructions=INSTRUCTION)

# Un caso difícil; cambia el id para mirar otros.
case = next(c for c in cases if c["id"] == "r67")
print(case["text"], "\nRuta correcta:", case["route"], "\n")
for agent, ref in ((decision_agent, decision_ref), (language_agent, model_ref)):
    row = (await run_routes(agent, [case], ref=ref, config=config))[0]
    print(f"{ref}: {row['route']} · confianza {row['confidence']} · {row['latency_ms']} ms")
    if row["probabilities"]:
        print("   ", {route: round(p, 3) for route, p in row["probabilities"].items()})

## 6. Correr los dos modelos sobre los 80 casos

Cada consulta es independiente y se mide su latencia, sus tokens y su costo. Con Gemini, `WAIT_S` espacia las consultas para respetar el límite gratuito de 15 por minuto: los 80 casos toman unos 6 minutos. Si alguna consulta falla, `retry_routes` repite solo esas.

In [ ]:
WAIT_S = 4.5 if model_ref.startswith("google:") else 0.0

decision_rows = await run_routes(decision_agent, cases, ref=decision_ref, config=config)
language_rows = await run_routes(language_agent, cases, ref=model_ref, config=config, wait_s=WAIT_S)

# Si hubo fallos, descomenta y ejecuta:
# decision_rows = await retry_routes(decision_agent, decision_rows, cases, ref=decision_ref, config=config)
# language_rows = await retry_routes(language_agent, language_rows, cases, ref=model_ref, config=config)

In [ ]:
def results_table(rows, label):
    """Una fila por caso, con la etiqueta, la ruta del modelo y si acertó."""
    table = pd.DataFrame(rows).set_index("id").join(gold[["route", "difficulty", "split"]], rsuffix="_gold")
    for column in ("confidence", "latency_ms", "estimated_cost_usd"):
        table[column] = pd.to_numeric(table[column])
    table["correct"] = table["route"] == table["route_gold"]
    table["system"] = label
    return table.reset_index()


results = pd.concat([results_table(decision_rows, "decision"), results_table(language_rows, "language")])
ok = results[results["status"] == "ok"]
summary = ok.groupby("system").agg(
    model=("model", "first"), cases=("correct", "size"), accuracy=("correct", "mean"),
    median_latency_ms=("latency_ms", "median"), p95_latency_ms=("latency_ms", lambda s: s.quantile(0.95)),
    cost_usd=("estimated_cost_usd", "sum"),
)
display(summary.round(3))
display(ok.pivot_table(index="difficulty", columns="system", values="correct", aggfunc="mean").round(3))

In [ ]:
# Matriz de confusión: filas, la ruta correcta; columnas, la que eligió el modelo.
for system in ("decision", "language"):
    part = ok[ok["system"] == system]
    print(system, part["model"].iloc[0])
    display(pd.crosstab(part["route_gold"], part["route"]))

## 7. Barrer el umbral con los casos `tune`

Un **umbral** es la confianza mínima para aceptar la ruta del modelo. Bajo el umbral, el caso se **deriva a una persona** (`needs_human`). Subirlo tiene un precio: se derivan más casos, pero los que se quedan con el modelo aciertan más.

Para cada umbral calculamos:

- `kept`: cuántos casos se quedan con el modelo.
- `derived_rate`: la proporción derivada a una persona.
- `accuracy_kept`: la proporción de aciertos entre los que se quedan.
- `errors_kept`: errores que pasan sin revisión, el número que más importa si un error es caro.

In [ ]:
THRESHOLDS = np.round(np.arange(0.0, 1.0001, 0.05), 2)


def sweep(table, thresholds=THRESHOLDS):
    rows = []
    for threshold in thresholds:
        kept = table[table["confidence"] >= threshold]
        rows.append({
            "threshold": threshold,
            "kept": len(kept),
            "derived_rate": round(1 - len(kept) / len(table), 3),
            "accuracy_kept": round(kept["correct"].mean(), 3) if len(kept) else None,
            "errors_kept": int((~kept["correct"]).sum()),
        })
    return pd.DataFrame(rows)


tune = ok[ok["split"] == "tune"]
sweeps = {system: sweep(tune[tune["system"] == system]) for system in ("decision", "language")}
display(pd.concat(sweeps, axis=1))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for system, table in sweeps.items():
    ax.plot(table["derived_rate"], table["accuracy_kept"], marker="o", label=system)
ax.set_xlabel("Proporción derivada a una persona")
ax.set_ylabel("Aciertos entre los que se quedan")
ax.set_title("Cada punto es un umbral (casos tune)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 8. Elegir un rango de umbral y comprobarlo con los casos `test`

Elijan **un modelo y un rango** de umbral, no un punto: con pocos casos, un cambio de una centésima mueve uno o dos casos de lado. Justifiquen con la tabla: cuántos errores aceptan que pasen sin revisión y qué proporción de mensajes puede revisar una persona.

Después lo comprobamos con los casos `test`, que no se usaron para elegir. Si allí el rango se comporta muy distinto, díganlo: es un resultado, no un error del laboratorio.

In [ ]:
CHOSEN_SYSTEM = "decision"  # @param ["decision", "language"]
THRESHOLD_LOW = 0.80  # @param {type: "number"}
THRESHOLD_HIGH = 0.85  # @param {type: "number"}

chosen = ok[ok["system"] == CHOSEN_SYSTEM]
check = pd.concat([
    sweep(chosen[chosen["split"] == split], [THRESHOLD_LOW, THRESHOLD_HIGH]).assign(split=split)
    for split in ("tune", "test")
])
display(check.set_index(["split", "threshold"]))

In [ ]:
# Los casos test con el umbral bajo: errores que pasan sin revisión y casos derivados.
test = chosen[chosen["split"] == "test"].join(gold["text"], on="id")
columns = ["id", "text", "route_gold", "route", "confidence", "difficulty"]
print("Errores que pasan sin revisión:")
display(test[(test["confidence"] >= THRESHOLD_LOW) & ~test["correct"]][columns])
print("Derivados a una persona:")
display(test[test["confidence"] < THRESHOLD_LOW].sort_values("confidence")[columns])

## 9. Probar con mensajes propios

Escriban **tres mensajes** como los escribiría un pasajero y anoten la ruta correcta según las reglas: uno claro, uno que toque dos temas y uno en que sea difícil decidir. Cuenten si el umbral que eligieron los deriva o no.

In [ ]:
MY_CASES = [
    {"id": "p01", "text": "", "route": "", "difficulty": "clear"},
    {"id": "p02", "text": "", "route": "", "difficulty": "hard"},
    {"id": "p03", "text": "", "route": "", "difficulty": "hard"},
]

own = [case for case in MY_CASES if case["text"].strip()]
invalid = [case["id"] for case in own if case["route"] not in Route.__args__]
if invalid:
    raise ValueError(f"Ruta inválida en {invalid}: usa una de {Route.__args__}.")
own_rows = []
for agent, ref in ((decision_agent, decision_ref), (language_agent, model_ref)):
    for row in await run_routes(agent, own, ref=ref, config=config, wait_s=WAIT_S if ref == model_ref else 0):
        expected = next(case["route"] for case in own if case["id"] == row["id"])
        own_rows.append({**row, "route_gold": expected, "correct": row["route"] == expected,
                         "derived": row["confidence"] is None or row["confidence"] < THRESHOLD_LOW})
display(pd.DataFrame(own_rows)[["id", "model", "route_gold", "route", "confidence", "derived"]] if own_rows
        else "Escriban sus mensajes en MY_CASES.")

## 10. Guardar la evidencia y decidir

Guardamos todas las respuestas en `decisions.csv`, los barridos en `sweep.csv` y, en `experiment.json`, los modelos, la instrucción, el esquema, el rango elegido, sus casos propios y sus decisiones. Con esos archivos otra persona puede recalcular el barrido sin consultar a ningún modelo.

Completa `DECISIONS` antes de guardar. Cada respuesta es texto libre, de dos a cuatro oraciones, entre las comillas. Citen cifras y casos concretos:

1. **Modelo**: cuál decide la ruta en su sistema y por qué, con aciertos, latencia y costo de la sección 6.
2. **Umbral**: el rango elegido, cuántos casos deriva y cuántos errores deja pasar en `tune` y en `test`.
3. **Error que pasa**: un caso que el modelo elegido respondió mal con confianza alta; qué lo confundió.
4. **Confianza declarada**: si la confianza del modelo de lenguaje separa aciertos de errores como la de d1, con la tabla del barrido.
5. **Casos propios**: qué mostró el más difícil.
6. **En su proyecto**: qué decisión de su proyecto podría tomar un modelo de decisión y cuánto costaría un error que pasa sin revisión.

Ejemplo para el umbral, con otras cifras: «d1 con umbral entre 0,75 y 0,80: en tune deriva el 22% y deja pasar 1 error de 32; en test deriva el 26% y deja pasar 2. Preferimos derivar uno de cada cuatro mensajes antes que mandar un reclamo de asistencia especial al equipo equivocado.»

Si no aparece alguna falla real, informa ese resultado; no inventes errores ni presentes salidas simuladas como mediciones.

In [ ]:
DECISIONS = {
    # 1. Qué modelo decide la ruta y por qué: aciertos, latencia y costo.
    "model": "",
    # 2. Rango de umbral elegido; derivados y errores que pasan, en tune y en test.
    "threshold": "",
    # 3. Un error con confianza alta: id, qué respondió, qué correspondía y qué lo confundió.
    "confident_error": "",
    # 4. ¿La confianza declarada del modelo de lenguaje separa aciertos de errores?
    "declared_confidence": "",
    # 5. Qué mostró el más difícil de sus mensajes propios.
    "own_cases": "",
    # 6. Qué decisión de su proyecto tomaría un modelo de decisión y cuánto costaría un error sin revisión.
    "project": "",
}

OUTPUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT / "decisions.csv", index=False)
pd.concat(sweeps, names=["system"]).reset_index(level=0).to_csv(OUTPUT / "sweep.csv", index=False)
payload = {
    "models": {"decision": decision_ref, "language": model_ref,
               "d1_quant": D1_QUANT if decision_ref == "d1:colab" else "servidor del curso"},
    "instruction": INSTRUCTION,
    "output_schema": {"decision": RouteDecision.model_json_schema(), "language": RouteAnswer.model_json_schema()},
    "dataset": routes["name"],
    "chosen": {"system": CHOSEN_SYSTEM, "threshold_low": THRESHOLD_LOW, "threshold_high": THRESHOLD_HIGH},
    "check": check.to_dict(orient="records"),
    "own_cases": MY_CASES,
    "own_results": own_rows,
    "decisions": DECISIONS,
}
save_experiment(OUTPUT / "experiment.json", payload, config, lab=4)
print("Evidencia guardada en", OUTPUT)

## 11. Descargar y entregar

Ejecuta la celda siguiente para descargar un ZIP con `decisions.csv`, `sweep.csv` y `experiment.json`. Guarda también el notebook **con sus salidas** mediante **Archivo → Descargar → Descargar .ipynb**. Si haces otra iteración, descarga antes la versión actual: cada ejecución sobrescribe los archivos.

**Producto del equipo:** notebook, `decisions.csv`, `sweep.csv`, `experiment.json`, sus mensajes propios y las decisiones. Después se incorporan a `labs/lab04-decision/` del repositorio del equipo; solo lo validado se promueve a `src/`. No hace falta usar Git desde Colab.

**Cada integrante:** reflexión propia, historial completo del asistente usado o declaración de que no usó uno, y enlace al producto compartido. Retira credenciales y datos sensibles e indica las omisiones.

**Canvas:** responde la pregunta individual de cinco minutos al cierre de la clase del 06/11. Producto, reflexión e historial se entregan hasta el **martes 10/11 a las 23:59, hora de Chile**.

**Extensiones opcionales** (celdas siguientes):

- **Calibración**: ¿cuando el modelo dice 0,8, acierta el 80% de las veces? Diagrama de confiabilidad y error de calibración esperado (ECE).
- **Cascada**: d1 decide; bajo el umbral, decide el modelo de lenguaje; si este también duda, `needs_human`.
- Repetir la sección 6 con `Q4_K_M`, con una instrucción sin las reglas o con otro modelo de decisión que sirva `/v1/systemone` en `llama-server` (por ejemplo, d1-3B), y comparar el barrido.

Referencias opcionales (inglés): [PydanticAI: modelos de decisión](https://pydantic.dev/docs/ai/models/decision/) y [System One](https://pydantic.dev/docs/ai/models/system-one/), unos 15 minutos.

In [ ]:
import shutil
from datetime import datetime, timezone

# Un nombre distinto conserva las iteraciones descargadas antes de sobrescribirlas.
archive_name = "lab04-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
archive_path = Path(shutil.make_archive(str(ROOT / archive_name), "zip", root_dir=ROOT / "labs", base_dir="lab04-decision"))
print("Descarga preparada:", archive_path.name)
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab, copia el ZIP desde la ruta indicada.")
else:
    files.download(str(archive_path))

### Extensión: calibración

Agrupamos los casos por tramos de confianza y comparamos la confianza media de cada tramo con su proporción de aciertos. Si los puntos caen sobre la diagonal, la confianza se puede leer como probabilidad de acertar. El **ECE** resume la distancia, ponderada por la cantidad de casos de cada tramo: 0 es perfecto. Con 80 casos, cada tramo tiene pocos y el resultado es ruidoso.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
for system in ("decision", "language"):
    bins, ece = reliability(ok[ok["system"] == system])
    ax.plot(bins["mean_confidence"], bins["accuracy"], marker="o", label=f"{system} · ECE {ece:.3f}")
    display(bins.round(3))
ax.set_xlabel("Confianza media del tramo")
ax.set_ylabel("Proporción de aciertos")
ax.legend()
plt.show()

### Extensión: cascada en tres pasos

1. d1 decide; si su confianza supera `THRESHOLD_LOW`, se usa su ruta.
2. Si no, decide el modelo de lenguaje, y se usa su ruta si su confianza declarada supera `LANGUAGE_THRESHOLD`.
3. Si ninguno está seguro, `needs_human`, con el paso que lo disparó.

Esta versión reutiliza las respuestas ya guardadas: no vuelve a consultar los modelos.

In [ ]:
LANGUAGE_THRESHOLD = 0.9  # @param {type: "number"}

decision = ok[ok["system"] == "decision"].set_index("id")
language = ok[ok["system"] == "language"].set_index("id").reindex(decision.index)
by_decision = decision["confidence"] >= THRESHOLD_LOW
by_language = ~by_decision & (language["confidence"] >= LANGUAGE_THRESHOLD)
cascade = decision[["route_gold", "split"]].copy()
cascade["decided_by"] = np.select([by_decision, by_language], ["decision", "language"], "needs_human")
cascade["route"] = np.select([by_decision, by_language], [decision["route"], language["route"]], None)
cascade["correct"] = cascade["route"] == cascade["route_gold"]
display(cascade.groupby("decided_by").agg(cases=("correct", "size"), accuracy=("correct", "mean")).round(3))